# QLoRA training — SCOTUS classifier (Kaggle)

Run this notebook on Kaggle:
1. Notebook settings → **Accelerator**: GPU (P100 or T4x2). **Internet**: On (needed to pull
   the base model/dataset from Hugging Face).
2. Add this repo to the notebook (upload as a Kaggle Dataset, or `git clone` it in a cell
   below if the repo is on GitHub).
3. Set `HF_TOKEN` as a Kaggle secret if you plan to `--push-to-hub` adapter checkpoints
   (recommended — Kaggle's interactive session state isn't durable across restarts, so
   pushing checkpoints as training progresses is the main mitigation for hitting the
   session time limit mid-run).
4. Run cells top to bottom. The unweighted and weighted runs are independent — run them in
   separate sessions if you're tight on the 30 GPU-hrs/week quota.

In [ ]:
# If the repo isn't already present in /kaggle/working, clone it here:
# !git clone <your-repo-url> repo && cd repo

!pip install -q -r requirements.txt

In [ ]:
import os

from kaggle_secrets import UserSecretsClient

try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    print("No HF_TOKEN secret found — push-to-hub checkpointing will be unavailable.")

from huggingface_hub import login

if os.environ.get("HF_TOKEN"):
    login(token=os.environ["HF_TOKEN"])

## Sanity check on a tiny subset

Highest-risk step (Day 2 of the plan): confirm the QLoRA + SEQ_CLS loop runs end-to-end
without OOM before committing GPU-hours to a full run.

In [ ]:
from src.data.load import load_scotus
from src.data.preprocess import tokenize_batch
from src.model.qlora import apply_lora, load_quantized_classifier
from transformers import AutoTokenizer

MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"

dataset = load_scotus()
small_train = dataset["train"].select(range(32))

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = load_quantized_classifier(MODEL_NAME, num_labels=14)
model = apply_lora(model)
model.print_trainable_parameters()

sample = tokenize_batch(small_train[:4], tokenizer, max_length=512)
print({k: len(v) for k, v in sample.items()})

## Full training runs

Unweighted baseline, then the class-weighted-loss run — the comparison between the two is
the evidence behind the imbalance-handling resume bullet.

In [ ]:
!python -m src.train.run_train --config configs/scotus_phi3.yaml

In [ ]:
!python -m src.train.run_train --config configs/scotus_phi3.yaml --weighted

Next: run `src/eval/metrics.py` against the held-out test split for both checkpoints and
`src/eval/benchmark.py` for latency/memory numbers (Day 5 of the plan).